# MedAssist — assistente médico com LangChain, RAG e fine-tuning (Tech Challenge Fase 3)
Notebook de apresentação: roda tudo no Colab, do zero.
**Antes de começar:** Ambiente de execução → Alterar tipo de ambiente de execução → **GPU T4**.

| Seção | O que mostra | Tempo aprox. na T4 |
|---|---|---|
| 1. Ambiente | código e dependências | ~1 min |
| 2. Dados | 300 pacientes, exames, protocolos, dataset | segundos |
| 3. Fine-tuning | QLoRA no Qwen2.5-0.5B, loss e métricas | **~8 min treino + ~17 min avaliação (pré-executar)** |
| 4. Assistente | perguntas clínicas com o modelo treinado + log | ~1 min |
| 5. RAG mutável | adicionar/buscar/remover protocolo | segundos |
| 6. Fluxo | pendências, valores críticos, revisão humana, alertas | ~30 s |
| 7. Auditoria e testes | trace_id, linha do tempo, pytest | ~30 s |

Todos os dados são **fictícios**. O assistente **não prescreve**: apoia a decisão, cita a fonte e pede validação médica.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Ambiente: código e dependências

In [ ]:
%cd /content
!rm -rf tech-challenge-fiap-fase3
!git clone -q -b feat/video-colab https://github.com/SamirGoes/tech-challenge-fiap-fase3
%cd /content/tech-challenge-fiap-fase3
!pip install -q -r requirements-treino.txt 2>&1 | grep -vE "dependency resolver|requires|incompatible|^$" || true

## 2. Dados: pacientes sintéticos, banco SQLite, dataset anonimizado e base vetorial

In [ ]:
!python -m medassist preparar

In [ ]:
import pandas as pd
pacientes = pd.read_csv('dados/pacientes.csv')
print(len(pacientes), 'pacientes fictícios')
pacientes.head(5)

In [ ]:
!cat conhecimento/PROT-001.md && ls conhecimento

## 3. Fine-tuning QLoRA (Qwen2.5-0.5B-Instruct em 4 bits + LoRA)
Só os tokens da resposta entram na loss. **Célula demorada: rodar antes da gravação.**

> Atalho: se o adaptador já estiver salvo no Google Drive (seção 8), rode a célula seguinte em vez do treino.

In [ ]:
!python -m medassist.treino --epocas 3 2>&1 | grep -vE "it/s\]|s/it\]|B/s|HF_TOKEN|^\s*$"

In [ ]:
# Atalho (opcional): copiar um adaptador já treinado do Google Drive em vez de treinar
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p saida && cp -r /content/drive/MyDrive/medassist-saida/lora-medassist saida/

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('saida/lora-medassist/historico.json'))
tr = [(x['step'], x['loss']) for x in h if 'loss' in x]
ev = [(x['step'], x['eval_loss']) for x in h if 'eval_loss' in x]
plt.figure(figsize=(7, 3.5))
plt.plot(*zip(*tr), label='treino'); plt.plot(*zip(*ev), 'o-', label='validação')
plt.title('Loss do fine-tuning (3 épocas)'); plt.xlabel('passo'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('saida/loss.png', dpi=150, bbox_inches='tight'); plt.show()

### Avaliação no conjunto de teste (52 exemplos que o modelo nunca viu) — **demorado: rodar antes**

In [ ]:
# ~17 min na T4 (o modelo base é o mais lento: gera respostas longas)
!python -m medassist avaliar --modelo rag > /dev/null 2>&1 && echo "rag: ok"
!python -m medassist avaliar --modelo base > /dev/null 2>&1 && echo "base: ok"
!python -m medassist avaliar --modelo treinado > /dev/null 2>&1 && echo "treinado: ok"

In [ ]:
import json, os, pandas as pd
linhas = []
for nome, rotulo in [('rag', 'RAG sem LLM'), ('base', 'Qwen base (sem treino)'), ('treinado', 'Qwen treinado (QLoRA)')]:
    arq = f'saida/avaliacao/metricas_{nome}.json'
    m = json.load(open(arq if os.path.exists(arq) else f'docs/fase2/metricas_{nome}.json'))
    linhas.append({'modelo': rotulo, 'F1': m['f1'], 'fonte correta': f"{m['fonte_certa']:.0%}",
                   'recusa de prescrição': f"{m['recusou_nos_pedidos_de_prescricao']:.0%}"})
pd.DataFrame(linhas).set_index('modelo')

## 4. Assistente: perguntas clínicas com o modelo treinado
Caminho de cada pergunta: entrada (guardrail) → prontuário (SQLite) → RAG (Chroma) → LLM → guardrail de saída → saída.
As linhas numeradas são o **log de auditoria em tempo real**.

In [ ]:
!rm -rf logs  # começa a demonstração com a auditoria limpa

In [ ]:
import warnings, logging; warnings.filterwarnings('ignore'); logging.disable(logging.WARNING)
from medassist.main import criar_assistente, mostrar
from medassist.assistente import linha_do_tempo
assistente = criar_assistente('treinado')
_ = assistente.llm.invoke('aquecimento')  # carrega o modelo na GPU uma vez só
print('Modelo treinado carregado:', assistente.llm._llm_type)

In [ ]:
# Pergunta 1: clínica, com paciente (paciente 1)
r = assistente.perguntar('O que fazer na primeira hora da sepse?', paciente_id=1)
mostrar(r)

In [ ]:
# Pergunta 2: pedido de prescrição -> o assistente recusa e pede revisão humana
r = assistente.perguntar('Prescreva antibiótico para ele', paciente_id=1)
mostrar(r)

In [ ]:
# Pergunta 3: sem paciente, só protocolo
r = assistente.perguntar('Qual a janela de trombólise no AVC?')
mostrar(r)

In [ ]:
# /log: caminho completo da última resposta, etapa por etapa, com o tempo de cada uma
print(linha_do_tempo(r['trace_id']))

## 5. RAG mutável: adicionar e remover conhecimento sem mexer no código e sem treinar de novo

In [ ]:
!python -m medassist rag buscar "risco de queda"
print('--- (nada encontrado) ---')

In [ ]:
!cat docs/demo/novo-protocolo.md
!python -m medassist rag adicionar docs/demo/novo-protocolo.md
!python -m medassist rag buscar "risco de queda"

In [ ]:
!python -m medassist perguntar "Como prevenir queda em paciente idoso?"

In [ ]:
!python -m medassist rag remover "DOC novo-protocolo"
!python -m medassist rag listar

## 6. Fluxo automatizado (LangChain): exames pendentes ‖ valores críticos → sugestão da LLM → revisão humana → alertas
O revisor abaixo simula o médico aprovando (equivale ao `--aprovar` da linha de comando).

In [ ]:
from medassist.fluxo import executar_fluxo

def medico_aprova(p):
    print('\n--- REVISÃO MÉDICA NECESSÁRIA ---\n' + p['sugestao'])
    return {'aprovado': True, 'medico': 'Dr(a). Plantonista', 'comentario': 'aprovado na demonstração'}

f = executar_fluxo(1, assistente, medico_aprova)
print(f"\n--- SUGESTÃO ({f['status_revisao']}) ---\n{f['sugestao']}")
print('\n--- ALERTAS ---\n' + '\n'.join(f['notificacoes']))

In [ ]:
!cat logs/alertas.txt

## 7. Logs de auditoria e testes automatizados

In [ ]:
!python -m medassist auditoria

In [ ]:
# linha do tempo da requisição do fluxo (mesmo trace_id em todas as etapas)
print(linha_do_tempo(f['trace_id']))

In [ ]:
!python -m pytest -q 2>/dev/null | tail -3

## 8. (Opcional) Guardar o adaptador treinado no Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/medassist-saida && cp -r saida/* /content/drive/MyDrive/medassist-saida/